## Milestone 2: Data Cleaning and Preparation

**File:** 

notebooks/02_data_cleaning_and_preparation.ipynb`

### Purpose

Load the eight raw files, fix data types, handle missing values and duplicates, validate the data against its reference tables, and save analysis-ready tables for the next notebook. Every step is recorded in a cleaning log.

### Inputs (`data/raw/`)

| File | Rows | Columns |
|---|---|---|
| sales_transactions.csv | 113,377 | 15 |
| customers.csv | 1,200 | 8 |
| stores.csv | 10 | 7 |
| products.csv | 15 | 6 |
| promotions.csv | 9 | 3 |
| 2019-population_census-report-per-county.csv | 47 | 11 |
| cbk_inflation rates.csv | 45 | 4 |
| kenya_public_holidays.csv | 98 | 3 |

### Outputs (data/processed/)

Eight cleaned tables as CSV (sales_clean.csv, customers_clean.csv, stores_clean.csv, products_clean.csv, promotions_clean.csv, census_clean.csv, inflation_clean.csv, holidays_clean.csv) and cleaning_log.csv. CSV was chosen over Parquet so the files open anywhere and need no extra packages. Date columns must be read back with parse_dates.

Cleaned shapes: sales (113,168 x 18), customers (1,200 x 8), stores (10 x 7), products (15 x 6), promotions (9 x 3), census (47 x 11), inflation (45 x 5), holidays (98 x 3).

### Cleaning steps

| Step | Table | What was done | Rows affected |
|---|---|---|---|
| 1. Types | sales | transaction_date text (month first) converted to datetime; range 2025-01-01 to 2026-10-31 | 113,377 converted, 0 failed |
| 1. Types | customers | signup_date`text (month first) converted to datetime; range 2024-01-01 to 2025-12-01 | 1,200 |
| 1. Types | stores | opening_date converted to datetime | 10 |
| 1. Types | holidays | Date converted to datetime | 98 |
| 1. Types | census | 8 numeric columns converted from text with commas to integers; total population 47,564,316 | 376 values |
| 1. Types | inflation | Year and Month name combined into a month_start date; 45 months, January 2023 to September 2026, no gaps or duplicates | 45 |
| 2. Missing | sales | blank customer_id set to GUEST, with an is_guest flag added | 5,735 (5.06%) |
| 2. Missing | sales | blank payment_method  set to Unknown | 2,300 (2.03%) |
| 3. Duplicates | sales | fully identical rows removed (first copy kept); 113,377 to 113,168 rows; other tables had none | 209 |
| 4. Validity | sales | promo IDs labelled with years 23 and 24 remapped to 25 and 26; original kept in promo_id_raw | 35,835 |
| 4. Outliers | sales | none removed; see checks below | 0 |
| 5. Integrity | sales | before_signup flag added (see data quality findings) | 12,959 flagged |

### Validation checks that passed

Unit price and unit cost match the product catalogue on every row.
Gross sales, discount, net sales, cost of goods sold and gross profit reconcile arithmetically.
Quantity ranges from 1 to 5, with no negative or zero values.
Every store_id exists in stores, and every non-guest customer_id exists in customers.
Discount amounts match the promotion percentages after the remap.
The 209 duplicates were true duplicates: the number of repeated transaction_id + product_id pairs equals the number of fully identical rows.

### Data quality findings and decisions

**Promotion labels.**
   
35,835 rows (31.6%) carried promo IDs labelled 2023 and 2024 that did not exist in promotions. Their sale dates and discount rates matched the 2025 and 2026 promotions exactly, so they were remapped.
After the remap, each promotion's sales fall entirely within its own month. PROMO_XMAS26 has no sales because the data ends in October 2026.

**Analysis window.** 

The data covers 1 January 2025 to 31 October 2026 (22 months). The earlier planned window of 2023 to 2024 does not apply to the sales data.

**Transactions dated before customer signup.**

12,959 sales rows (11.5% of all rows, 12.1% of rows from known customers) are dated before the customer's signup_date. They affect 561 of 1,200 customers, all of whom signed up in 2025 (584 customers signed up in 2025 and 616 in 2024). About 24.6% of transactions by 2025 signups fall before signup, by up to 333 days (median 96). Customers who signed up in 2024 have no early rows, because all of them signed up before the first transaction date. Transaction dates appear to have been generated independently of signup dates. 
**Decision:** 
signup_date is not used for tenure or cohorts. Customer tenure is derived from the first observed purchase instead. The rows are kept and flagged, not dropped, so revenue and demand totals are unaffected.

**Guest transactions.** 

5,735 rows (5.1%) have no customer ID and are excluded from customer-level analysis (retention, RFM), but kept for sales, store, product and forecasting analysis.

**Unknown payment method.** 

2,300 rows are kept as their own Unknown category in payment-channel comparisons, not merged into another method.

### Limitations to carry forward

- Inflation data ends in September 2026, but sales run to October 2026.
- The holidays file ends on 20 October 2026.
- First observed purchase cannot be earlier than 1 January 2025, so customers near the start of the window may appear newer than they are.
- There is one year of history for each calendar month except January to October, which limits seasonal modelling.


In [1]:
import pandas as pd

raw = "../data/raw/"

sales = pd.read_csv(raw + "sales_transactions.csv")
customers = pd.read_csv(raw + "customers.csv")
stores = pd.read_csv(raw + "stores.csv")
products = pd.read_csv(raw + "products.csv")
census = pd.read_csv(raw + "2019-population_census-report-per-county.csv")
promotions = pd.read_csv(raw + "promotions.csv")
inflation = pd.read_csv(raw + "cbk_inflation rates.csv")
holidays = pd.read_csv(raw + "kenya_public_holidays.csv")

tables = {"sales": sales, "customers": customers, "stores": stores, "products": products,
        "promotions": promotions, "census": census, "inflation": inflation, 
         "holidays": holidays,}

for name, df in tables.items():
    print(f"{name}:{df.shape}")

sales:(113377, 15)
customers:(1200, 8)
stores:(10, 7)
products:(15, 6)
promotions:(9, 3)
census:(47, 11)
inflation:(45, 4)
holidays:(98, 3)


In [2]:
### the below log will keep track record of data cleaning steps
log = []

def record(step, table, rows_before, rows_after, fixed, note=""):
    log.append({
        "step": step,
        "table": table,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "values_fixed": fixed,
        "note": note,
    })

for name, df in tables.items():
    record("0. raw load", name, len(df), len(df), 0, f"{df.shape[1]} columns")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns


In [3]:
sales.head()

,transaction_id,transaction_date,customer_id,store_id,product_id,quantity,unit_cost_kes,unit_price_kes,gross_sales_kes,promo_id,discount_amount_kes,net_sales_kes,cogs_kes,gross_profit_kes,payment_method
0,TXN_100001,1/1/2025,CUST_00875,STR_003,PRD_013,1,130,185,185,PROMO_NONE,0.0,185.0,130,55.0,M-Pesa
1,TXN_100001,1/1/2025,CUST_00875,STR_003,PRD_009,1,90,130,130,PROMO_NONE,0.0,130.0,90,40.0,M-Pesa
2,TXN_100001,1/1/2025,CUST_00875,STR_003,PRD_011,1,140,200,200,PROMO_NONE,0.0,200.0,140,60.0,M-Pesa
3,TXN_100002,1/1/2025,CUST_00664,STR_010,PRD_001,3,160,210,630,PROMO_NONE,0.0,630.0,480,150.0,M-Pesa
4,TXN_100002,1/1/2025,CUST_00664,STR_010,PRD_012,1,230,320,320,PROMO_NONE,0.0,320.0,230,90.0,M-Pesa


In [4]:
sales.dtypes

transaction_id             str
transaction_date           str
customer_id                str
store_id                   str
product_id                 str
quantity                 int64
unit_cost_kes            int64
unit_price_kes           int64
gross_sales_kes          int64
promo_id                   str
discount_amount_kes    float64
net_sales_kes          float64
cogs_kes                 int64
gross_profit_kes       float64
payment_method             str
dtype: object

In [5]:
### Converting Transaction Date from String to Datetime


In [6]:
sales_clean = sales.copy()

print(repr(sales["transaction_date"].iloc[0]))

sales_clean["transaction_date"] = pd.to_datetime(
    sales["transaction_date"].str.strip(),
    dayfirst=False,
    errors="coerce"
)

converted = sales_clean["transaction_date"].notna().sum()
failed = sales_clean["transaction_date"].isna().sum()

print("Converted:", converted)
print("Failed:", failed)
print("Range:", sales_clean["transaction_date"].min(), "to", sales_clean["transaction_date"].max())

'1/1/2025'
Converted: 113377
Failed: 0
Range: 2025-01-01 00:00:00 to 2026-10-31 00:00:00


In [7]:
print("raw   :", sales["transaction_date"].dtype)
print("clean :", sales_clean["transaction_date"].dtype)
print()

raw   : str
clean : datetime64[us]



In [8]:
record(
    "1. types", "sales",
    len(sales), len(sales_clean),
    converted,
    "transaction_date text; other columns already correct"
)

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...


In [9]:
for name in ["customers", "stores", "products", "promotions", "census", "inflation", "holidays"]:
    print("-----", name)
    print(tables[name].dtypes)

----- customers
customer_id                   str
customer_name                 str
gender                        str
age                         int64
county                        str
signup_date                   str
preferred_payment_method      str
loyalty_tier                  str
dtype: object
----- stores
store_id          str
store_name        str
county            str
sub_county        str
store_format      str
sqft            int64
opening_date      str
dtype: object
----- products
product_id          str
product_name        str
category            str
sub_category        str
unit_cost_kes     int64
unit_price_kes    int64
dtype: object
----- promotions
promo_id          str
promo_name        str
discount_pct    int64
dtype: object
----- census
County                        str
Total_Population19            str
Male populatio 2019           str
Female population 2019        str
Households                    str
Av_HH_Size                  int64
LandArea                      

In [10]:
print("signup_date :", tables["customers"]["signup_date"].head(5).tolist())
print("opening_date:", tables["stores"]["opening_date"].head(5).tolist())
print("holidays    :", tables["holidays"]["Date"].head(5).tolist())
print()
print(tables["census"][["County", "Total_Population19", "Households", "LandArea", "Population Density", "Pop_change"]].head(5))

signup_date : ['4/12/2025', '11/5/2024', '7/15/2025', '5/20/2025', '12/6/2024']
opening_date: ['3/15/2019', '6/1/2020', '1/10/2021', '11/20/2019', '8/14/2021']
holidays    : ['10/20/2026', '10/10/2026', '6/1/2026', '5/27/2026', '5/1/2026']

            County Total_Population19 Households LandArea Population Density  \
0          Baringo            666,763    142,518   10,976                 61   
1            Bomet            875,689    187,641    2,531                346   
2          Bungoma          1,670,570    358,796    3,024                552   
3            Busia            893,681    198,152    1,696                527   
4  Elgeyo-Marakwet            454,480     99,861    3,032                150   

  Pop_change  
0    111,202  
1    151,503  
2     39,636  
3    405,606  
4     84,482  


In [11]:
customers_clean = customers.copy()
stores_clean = stores.copy()
holidays_clean = holidays.copy()

customers_clean["signup_date"] = pd.to_datetime(
    customers_clean["signup_date"], format="%m/%d/%Y", errors="raise"
)
stores_clean["opening_date"] = pd.to_datetime(
    stores_clean["opening_date"], format="%m/%d/%Y", errors="coerce"
)
holidays_clean["Date"] = pd.to_datetime(
    holidays_clean["Date"], format="%m/%d/%Y", errors="coerce"
)

checks = [
    ("customers", customers_clean, "signup_date"),
    ("stores", stores_clean, "opening_date"),
    ("holidays", holidays_clean, "Date"),
]

for name, df, col in checks:
    ok = df[col].notna().sum()
    bad = df[col].isna().sum()
    print(f"{name}.{col}: {ok} converted, {bad} failed, "
          f"{df[col].min().date()} to {df[col].max().date()}")

customers.signup_date: 1200 converted, 0 failed, 2024-01-01 to 2025-12-01
stores.opening_date: 10 converted, 0 failed, 2019-03-15 to 2022-03-22
holidays.Date: 98 converted, 0 failed, 2020-01-01 to 2026-10-20


In [12]:
record("1. types", "customers", len(customers), len(customers_clean),
       customers_clean["signup_date"].notna().sum(),
       "signup_date text (month first) -> datetime")

record("1. types", "stores", len(stores), len(stores_clean),
       stores_clean["opening_date"].notna().sum(),
       "opening_date text (month first) -> datetime")

record("1. types", "holidays", len(holidays), len(holidays_clean),
       holidays_clean["Date"].notna().sum(),
       "Date text (month first) -> datetime")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [13]:
census_clean = census.copy()

num_cols = ["Total_Population19", "Male populatio 2019", "Female population 2019",
            "Households", "LandArea", "Population Density",
            "Population in 2009", "Pop_change"]

for col in num_cols:
    census_clean[col] = pd.to_numeric(
        census_clean[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

print(census_clean[num_cols].dtypes)
print()
print("missing after conversion:")
print(census_clean[num_cols].isna().sum())
print()
print("total population:", census_clean["Total_Population19"].sum())

Total_Population19        int64
Male populatio 2019       int64
Female population 2019    int64
Households                int64
LandArea                  int64
Population Density        int64
Population in 2009        int64
Pop_change                int64
dtype: object

missing after conversion:
Total_Population19        0
Male populatio 2019       0
Female population 2019    0
Households                0
LandArea                  0
Population Density        0
Population in 2009        0
Pop_change                0
dtype: int64

total population: 47564316


In [14]:
record("1. types", "census", len(census), len(census_clean),
       len(num_cols) * len(census_clean),
       "8 numeric columns: text with commas -> int; total population 47,564,316")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [15]:
### looking for missing values in the clean files

clean = {
    "sales": sales_clean, "customers": customers_clean, "stores": stores_clean,
    "products": products, "promotions": promotions,
    "census": census_clean, "inflation": inflation, "holidays": holidays_clean,
}

for name, df in clean.items():
    missing = df.isna().sum()
    print("-----", name, df.shape)
    if missing.sum() == 0:
        print("no missing values")
    else:
        print(missing[missing > 0])

----- sales (113377, 15)
customer_id       5735
payment_method    2300
dtype: int64
----- customers (1200, 8)
no missing values
----- stores (10, 7)
no missing values
----- products (15, 6)
no missing values
----- promotions (9, 3)
no missing values
----- census (47, 11)
no missing values
----- inflation (45, 4)
no missing values
----- holidays (98, 3)
no missing values


In [16]:
s = sales_clean

print("share missing:")
print((s[["customer_id", "payment_method"]].isna().mean() * 100).round(2))

print()
print("payment_method values (including blanks):")
print(s["payment_method"].value_counts(dropna=False))

print()
print("rows missing BOTH:", (s["customer_id"].isna() & s["payment_method"].isna()).sum())

print()
print("missing customer_id by store:")
print(s[s["customer_id"].isna()]["store_id"].value_counts())

print()
print("blank-looking text in customer_id:",
      s["customer_id"].astype(str).str.strip().isin(["", "nan", "None", "NA", "N/A", "Unknown", "Guest"]).sum())

share missing:
customer_id       5.06
payment_method    2.03
dtype: float64

payment_method values (including blanks):
payment_method
M-Pesa    75699
Cash      24429
Card      10949
NaN        2300
Name: count, dtype: int64

rows missing BOTH: 138

missing customer_id by store:
store_id
STR_008    623
STR_007    601
STR_002    591
STR_005    583
STR_009    569
STR_003    562
STR_001    560
STR_010    552
STR_004    550
STR_006    544
Name: count, dtype: int64

blank-looking text in customer_id: 0


In [17]:
before_cust = sales_clean["customer_id"].isna().sum()
before_pay = sales_clean["payment_method"].isna().sum()

sales_clean["is_guest"] = sales_clean["customer_id"].isna()
sales_clean["customer_id"] = sales_clean["customer_id"].fillna("GUEST")
sales_clean["payment_method"] = sales_clean["payment_method"].fillna("Unknown")

print("customer_id missing:", before_cust, "->", sales_clean["customer_id"].isna().sum())
print("payment_method missing:", before_pay, "->", sales_clean["payment_method"].isna().sum())
print("rows:", len(sales_clean))

customer_id missing: 5735 -> 0
payment_method missing: 2300 -> 0
rows: 113377


In [18]:
record("2. missing", "sales", len(sales), len(sales_clean), before_cust,
       "customer_id blank -> 'GUEST' (is_guest flag added)")
record("2. missing", "sales", len(sales), len(sales_clean), before_pay,
       "payment_method blank -> 'Unknown'")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [19]:
print(sales_clean["payment_method"].value_counts())
print()
print(sales_clean["is_guest"].value_counts())

payment_method
M-Pesa     75699
Cash       24429
Card       10949
Unknown     2300
Name: count, dtype: int64

is_guest
False    107642
True       5735
Name: count, dtype: int64


In [20]:
### checking for dublicates

clean = {
    "sales": sales_clean, "customers": customers_clean, "stores": stores_clean,
    "products": products, "promotions": promotions,
    "census": census_clean, "inflation": inflation, "holidays": holidays_clean,
}

print("fully identical rows per table:")
for name, df in clean.items():
    print(f"  {name}: {df.duplicated().sum()}")

print()
print("sales: same transaction_id + product_id more than once:",
      sales_clean.duplicated(subset=["transaction_id", "product_id"]).sum())

print("customers: repeated customer_id:", customers_clean["customer_id"].duplicated().sum())
print("stores: repeated store_id:", stores_clean["store_id"].duplicated().sum())
print("products: repeated product_id:", products["product_id"].duplicated().sum())
print("promotions: repeated promo_id:", promotions["promo_id"].duplicated().sum())

fully identical rows per table:
  sales: 209
  customers: 0
  stores: 0
  products: 0
  promotions: 0
  census: 0
  inflation: 0
  holidays: 0

sales: same transaction_id + product_id more than once: 209
customers: repeated customer_id: 0
stores: repeated store_id: 0
products: repeated product_id: 0
promotions: repeated promo_id: 0


In [21]:
dup_mask = sales_clean.duplicated(keep=False)
dups = sales_clean[dup_mask].sort_values(["transaction_id", "product_id"])

print("rows involved (originals + copies):", len(dups))
print("transactions affected:", dups["transaction_id"].nunique())
print("quantity of the duplicated lines:")
print(dups["quantity"].value_counts().sort_index())
print()
dups.head(6)

rows involved (originals + copies): 418
transactions affected: 208
quantity of the duplicated lines:
quantity
1    214
2    118
3     46
4     20
5     20
Name: count, dtype: int64



,transaction_id,transaction_date,customer_id,store_id,product_id,quantity,unit_cost_kes,unit_price_kes,gross_sales_kes,promo_id,discount_amount_kes,net_sales_kes,cogs_kes,gross_profit_kes,payment_method,is_guest
413,TXN_100215,2025-01-03,CUST_00088,STR_007,PRD_012,1,230,320,320,PROMO_NONE,0.0,320.0,230,90.0,M-Pesa,False
528,TXN_100215,2025-01-03,CUST_00088,STR_007,PRD_012,1,230,320,320,PROMO_NONE,0.0,320.0,230,90.0,M-Pesa,False
633,TXN_100330,2025-01-04,CUST_00228,STR_008,PRD_007,1,30,50,50,PROMO_NONE,0.0,50.0,30,20.0,Cash,False
680,TXN_100330,2025-01-04,CUST_00228,STR_008,PRD_007,1,30,50,50,PROMO_NONE,0.0,50.0,30,20.0,Cash,False
878,TXN_100467,2025-01-06,CUST_00457,STR_010,PRD_004,1,48,65,65,PROMO_NONE,0.0,65.0,48,17.0,M-Pesa,False
946,TXN_100467,2025-01-06,CUST_00457,STR_010,PRD_004,1,48,65,65,PROMO_NONE,0.0,65.0,48,17.0,M-Pesa,False


In [22]:
### removing dublicates idenfied in sales file

before = len(sales_clean)

sales_clean = sales_clean.drop_duplicates(keep="first").reset_index(drop=True)

after = len(sales_clean)
removed = before - after

print("rows before:", before)
print("rows after :", after)
print("removed    :", removed)
print("identical rows left:", sales_clean.duplicated().sum())

rows before: 113377
rows after : 113168
removed    : 209
identical rows left: 0


In [23]:
record("3. duplicates", "sales", before, after, removed,
       "209 fully identical rows removed (kept first copy); other 7 tables had none")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [24]:
### Looking for outliers in my datasets

s = sales_clean

print("summary of numeric columns:")
print(s[["quantity", "unit_cost_kes", "unit_price_kes", "gross_sales_kes",
         "discount_amount_kes", "net_sales_kes", "cogs_kes", "gross_profit_kes"]].describe().round(1))

print()
print("arithmetic checks (rows that break the rule):")
print("  gross != quantity x price      :", (s["gross_sales_kes"] != s["quantity"] * s["unit_price_kes"]).sum())
print("  net != gross - discount        :", ((s["net_sales_kes"] - (s["gross_sales_kes"] - s["discount_amount_kes"])).abs() > 0.01).sum())
print("  cogs != quantity x unit cost   :", (s["cogs_kes"] != s["quantity"] * s["unit_cost_kes"]).sum())
print("  profit != net - cogs           :", ((s["gross_profit_kes"] - (s["net_sales_kes"] - s["cogs_kes"])).abs() > 0.01).sum())

print()
print("negative values:")
print("  quantity <= 0      :", (s["quantity"] <= 0).sum())
print("  net_sales < 0      :", (s["net_sales_kes"] < 0).sum())
print("  discount < 0       :", (s["discount_amount_kes"] < 0).sum())
print("  price below cost   :", (s["unit_price_kes"] < s["unit_cost_kes"]).sum())

summary of numeric columns:
       quantity  unit_cost_kes  unit_price_kes  gross_sales_kes  \
count  113168.0       113168.0        113168.0         113168.0   
mean        1.8          142.5           196.0            346.4   
std         1.0          115.4           149.6            370.6   
min         1.0           30.0            50.0             50.0   
25%         1.0           70.0           110.0            130.0   
50%         1.0          110.0           160.0            210.0   
75%         2.0          160.0           210.0            400.0   
max         5.0          520.0           680.0           3400.0   

       discount_amount_kes  net_sales_kes  cogs_kes  gross_profit_kes  
count             113168.0       113168.0  113168.0          113168.0  
mean                  14.8          331.6     251.9              79.7  
std                   37.0          356.1     280.8              82.6  
min                    0.0           40.0      30.0               1.0  
25%     

In [25]:
s = sales_clean

m = s.merge(products[["product_id", "unit_cost_kes", "unit_price_kes"]],
            on="product_id", how="left", suffixes=("", "_cat"))
print("product_id not in catalogue :", m["unit_price_kes_cat"].isna().sum())
print("price differs from catalogue:", (m["unit_price_kes"] != m["unit_price_kes_cat"]).sum())
print("cost differs from catalogue :", (m["unit_cost_kes"] != m["unit_cost_kes_cat"]).sum())

p = s.merge(promotions[["promo_id", "discount_pct"]], on="promo_id", how="left")
print()
print("promo_id not in promotions  :", p["discount_pct"].isna().sum())
expected = (p["gross_sales_kes"] * p["discount_pct"] / 100).round(2)
print("discount differs from promo % :", ((p["discount_amount_kes"] - expected).abs() > 0.01).sum())

print()
print("store_id not in stores      :", (~s["store_id"].isin(stores["store_id"])).sum())
reg = s[s["customer_id"] != "GUEST"]
print("customer_id not in customers:", (~reg["customer_id"].isin(customers["customer_id"])).sum())

product_id not in catalogue : 0
price differs from catalogue: 0
cost differs from catalogue : 0

promo_id not in promotions  : 35835
discount differs from promo % : 0

store_id not in stores      : 0
customer_id not in customers: 0


In [26]:
s = sales_clean

unmatched = s[~s["promo_id"].isin(promotions["promo_id"])]

print("promo_ids not in promotions table:")
print(unmatched["promo_id"].value_counts(dropna=False))
print()
print("discount on those rows: min", unmatched["discount_amount_kes"].min(),
      "| max", unmatched["discount_amount_kes"].max())
print()
print("promotions table:")
print(promotions)

promo_ids not in promotions table:
promo_id
PROMO_XMAS23        5331
PROMO_SCHOOL24      5251
PROMO_SCHOOL23      5141
PROMO_EASTER23      5085
PROMO_MADARAKA23    5084
PROMO_EASTER24      4979
PROMO_MADARAKA24    4964
Name: count, dtype: int64

discount on those rows: min 5.0 | max 680.0

promotions table:
           promo_id                   promo_name  discount_pct
0        PROMO_NONE                 No Promotion             0
1    PROMO_EASTER25    Easter Super Savings 2025            10
2  PROMO_MADARAKA25      Madaraka Day Deals 2025            15
3    PROMO_SCHOOL25  Back to School Bonanza 2025            12
4      PROMO_XMAS25       Festive Mega Sale 2025            20
5    PROMO_EASTER26    Easter Super Savings 2026            10
6  PROMO_MADARAKA26      Madaraka Day Deals 2026            15
7    PROMO_SCHOOL26  Back to School Bonanza 2026            12
8      PROMO_XMAS26       Festive Mega Sale 2026            20


In [27]:
s = sales_clean
s_un = s[~s["promo_id"].isin(promotions["promo_id"])].copy()
s_un["implied_pct"] = (s_un["discount_amount_kes"] / s_un["gross_sales_kes"] * 100).round(1)

print("transactions with these promo_ids, by date range and implied discount %:")
print(s_un.groupby("promo_id").agg(
    rows=("promo_id", "size"),
    first_sale=("transaction_date", "min"),
    last_sale=("transaction_date", "max"),
    pct_min=("implied_pct", "min"),
    pct_max=("implied_pct", "max"),
))

print()
print("matched promo_ids in sales:")
ok = s[s["promo_id"].isin(promotions["promo_id"])]
print(ok.groupby("promo_id").agg(
    rows=("promo_id", "size"),
    first_sale=("transaction_date", "min"),
    last_sale=("transaction_date", "max"),
))

transactions with these promo_ids, by date range and implied discount %:
                  rows first_sale  last_sale  pct_min  pct_max
promo_id                                                      
PROMO_EASTER23    5085 2025-04-01 2025-04-30     10.0     10.0
PROMO_EASTER24    4979 2026-04-01 2026-04-30     10.0     10.0
PROMO_MADARAKA23  5084 2025-06-01 2025-06-30     15.0     15.0
PROMO_MADARAKA24  4964 2026-06-01 2026-06-30     15.0     15.0
PROMO_SCHOOL23    5141 2025-08-01 2025-08-31     12.0     12.0
PROMO_SCHOOL24    5251 2026-08-01 2026-08-31     12.0     12.0
PROMO_XMAS23      5331 2025-12-01 2025-12-31     20.0     20.0

matched promo_ids in sales:
             rows first_sale  last_sale
promo_id                               
PROMO_NONE  77333 2025-01-01 2026-10-31


In [28]:
promo_map = {
    "PROMO_EASTER23": "PROMO_EASTER25",
    "PROMO_MADARAKA23": "PROMO_MADARAKA25",
    "PROMO_SCHOOL23": "PROMO_SCHOOL25",
    "PROMO_XMAS23": "PROMO_XMAS25",
    "PROMO_EASTER24": "PROMO_EASTER26",
    "PROMO_MADARAKA24": "PROMO_MADARAKA26",
    "PROMO_SCHOOL24": "PROMO_SCHOOL26",
}

sales_clean["promo_id_raw"] = sales_clean["promo_id"]
remapped = sales_clean["promo_id"].isin(promo_map.keys()).sum()
sales_clean["promo_id"] = sales_clean["promo_id"].replace(promo_map)

print("rows remapped:", remapped)

p = sales_clean.merge(promotions[["promo_id", "discount_pct"]], on="promo_id", how="left")
expected = p["gross_sales_kes"] * p["discount_pct"] / 100

print("rows:", len(p))
print("promo_id not in promotions:", p["discount_pct"].isna().sum())
print("discount differs from promo %:", ((p["discount_amount_kes"] - expected).abs() > 0.01).sum())

rows remapped: 35835
rows: 113168
promo_id not in promotions: 0
discount differs from promo %: 0


In [29]:
record("4. validity", "sales", len(sales_clean), len(sales_clean), remapped,
       "7 promo_ids with year labels 23/24 remapped to 25/26 (dates and discount rates match); original kept in promo_id_raw")

record("4. outliers", "sales", len(sales_clean), len(sales_clean), 0,
       "no outliers removed: quantity 1-5, prices/costs match catalogue, arithmetic and discounts reconcile")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [30]:
chk = sales_clean[sales_clean["customer_id"] != "GUEST"].merge(
    customers_clean[["customer_id", "signup_date"]], on="customer_id", how="left")

before_signup = chk["transaction_date"] < chk["signup_date"]

print("transactions by known customers:", len(chk))
print("dated before the customer's signup_date:", before_signup.sum())
print(f"share: {before_signup.mean() * 100:.1f}%")

transactions by known customers: 107447
dated before the customer's signup_date: 12959
share: 12.1%


In [31]:
chk["days_before"] = (chk["signup_date"] - chk["transaction_date"]).dt.days
early = chk[before_signup]

print("customers affected:", early["customer_id"].nunique(), "of", customers_clean["customer_id"].nunique())
print()
print("how many days before signup (early rows only):")
print(early["days_before"].describe().round(0))
print()
print("early share by signup year:")
print(chk.groupby(chk["signup_date"].dt.year)
         .apply(lambda g: (g["transaction_date"] < g["signup_date"]).mean() * 100)
         .round(1))
print()
print("customers with signup in 2025 vs 2024:",
      (customers_clean["signup_date"].dt.year == 2025).sum(), "/",
      (customers_clean["signup_date"].dt.year == 2024).sum())

customers affected: 561 of 1200

how many days before signup (early rows only):
count    12959.0
mean       110.0
std         78.0
min          1.0
25%         44.0
50%         96.0
75%        165.0
max        333.0
Name: days_before, dtype: float64

early share by signup year:
signup_date
2024     0.0
2025    24.6
dtype: float64

customers with signup in 2025 vs 2024: 584 / 616


In [32]:
signup_lookup = customers_clean.set_index("customer_id")["signup_date"]
signup_for_row = sales_clean["customer_id"].map(signup_lookup)

sales_clean["before_signup"] = sales_clean["transaction_date"] < signup_for_row

flagged = sales_clean["before_signup"].sum()

print("rows:", len(sales_clean))
print("flagged before_signup:", flagged)
print("guest rows flagged (should be 0):",
      sales_clean.loc[sales_clean["is_guest"], "before_signup"].sum())

rows: 113168
flagged before_signup: 12959
guest rows flagged (should be 0): 0


In [33]:
print("transactions:", sales_clean["transaction_date"].min(), "to", sales_clean["transaction_date"].max())
print("signups:", customers_clean["signup_date"].min(), "to", customers_clean["signup_date"].max())
print(customers_clean.groupby(customers_clean["signup_date"].dt.to_period("M")).size())

transactions: 2025-01-01 00:00:00 to 2026-10-31 00:00:00
signups: 2024-01-01 00:00:00 to 2025-12-01 00:00:00
signup_date
2024-01    65
2024-02    42
2024-03    50
2024-04    46
2024-05    60
2024-06    49
2024-07    52
2024-08    57
2024-09    49
2024-10    50
2024-11    48
2024-12    48
2025-01    54
2025-02    49
2025-03    49
2025-04    64
2025-05    58
2025-06    42
2025-07    67
2025-08    54
2025-09    45
2025-10    42
2025-11    58
2025-12     2
Freq: M, dtype: int64


In [34]:
print(sales_clean["transaction_date"].dt.to_period("Y").value_counts().sort_index())
print(sales_clean.groupby("promo_id")["transaction_date"].agg(["min","max","count"]))

transaction_date
2025    61715
2026    51453
Freq: Y-DEC, Name: count, dtype: int64
                        min        max  count
promo_id                                     
PROMO_EASTER25   2025-04-01 2025-04-30   5085
PROMO_EASTER26   2026-04-01 2026-04-30   4979
PROMO_MADARAKA25 2025-06-01 2025-06-30   5084
PROMO_MADARAKA26 2026-06-01 2026-06-30   4964
PROMO_NONE       2025-01-01 2026-10-31  77333
PROMO_SCHOOL25   2025-08-01 2025-08-31   5141
PROMO_SCHOOL26   2026-08-01 2026-08-31   5251
PROMO_XMAS25     2025-12-01 2025-12-31   5331


In [35]:
record("5. integrity", "sales", len(sales_clean), len(sales_clean), int(flagged),
       "before_signup flag added: transactions dated before customer signup_date (561 customers, all 2025 signups); "
       "signup_date not used for tenure, first observed purchase used instead")

pd.DataFrame(log)

,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [36]:
print(inflation.dtypes)
print(inflation.head())

Year                          int64
Month                           str
Annual Average Inflation    float64
12-Month Inflation          float64
dtype: object
   Year      Month  Annual Average Inflation  12-Month Inflation
0  2026  September                      5.44                6.76
1  2026     August                      5.25                6.59
2  2026       July                      5.08                6.49
3  2026       June                      4.88                6.41
4  2026        May                      4.66                6.68


In [37]:
inflation_clean = inflation.copy()

inflation_clean["month_start"] = pd.to_datetime(
    inflation_clean["Year"].astype(str) + " " + inflation_clean["Month"].str.strip(),
    format="%Y %B",
    errors="raise",
)

inflation_clean = inflation_clean.sort_values("month_start").reset_index(drop=True)

print(inflation_clean["month_start"].min().date(), "to", inflation_clean["month_start"].max().date())
print("rows:", len(inflation_clean))
print("duplicate months:", inflation_clean["month_start"].duplicated().sum())
print("missing months:",
      len(pd.date_range(inflation_clean["month_start"].min(),
                        inflation_clean["month_start"].max(), freq="MS")) - len(inflation_clean))

record("1. types", "inflation", len(inflation), len(inflation_clean),
       len(inflation_clean),
       "Year + Month name -> month_start datetime (first of month); sorted oldest to newest")

pd.DataFrame(log)

2023-01-01 to 2026-09-01
rows: 45
duplicate months: 0
missing months: 0


,step,table,rows_before,rows_after,values_fixed,note
0,0. raw load,sales,113377,113377,0,15 columns
1,0. raw load,customers,1200,1200,0,8 columns
2,0. raw load,stores,10,10,0,7 columns
3,0. raw load,products,15,15,0,6 columns
4,0. raw load,promotions,9,9,0,3 columns
5,0. raw load,census,47,47,0,11 columns
6,0. raw load,inflation,45,45,0,4 columns
7,0. raw load,holidays,98,98,0,3 columns
8,1. types,sales,113377,113377,113377,transaction_date text; other columns already c...
9,1. types,customers,1200,1200,1200,signup_date text (month first) -> datetime


In [38]:
import os
os.makedirs("../data/processed", exist_ok=True)

processed = {
    "sales": sales_clean,
    "customers": customers_clean,
    "stores": stores_clean,
    "products": products,
    "promotions": promotions,
    "census": census_clean,
    "inflation": inflation_clean,
    "holidays": holidays_clean,
}

for name, df in processed.items():
    df.to_csv(f"../data/processed/{name}_clean.csv", index=False)
    print(f"{name}_clean saved: {df.shape}")

log_df = pd.DataFrame(log)
log_df.to_csv("../data/processed/cleaning_log.csv", index=False)
print("cleaning_log saved:", log_df.shape)

sales_clean saved: (113168, 18)
customers_clean saved: (1200, 8)
stores_clean saved: (10, 7)
products_clean saved: (15, 6)
promotions_clean saved: (9, 3)
census_clean saved: (47, 11)
inflation_clean saved: (45, 5)
holidays_clean saved: (98, 3)
cleaning_log saved: (20, 6)
